In [6]:
# =================================================================
# PROYECTO INTEGRADOR: MACHINE LEARNING - FASE 2
# GRUPO 2: Análisis de Satisfacción en Productos de Oficina
# Notebook 2: Modelado Clásico, AutoML y Análisis de Error
# =================================================================

# --- BLOQUE 1: IMPORTACIÓN DE LIBRERÍAS ---
import pandas as pd
import numpy as np
import gc
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import xgboost as xgb

In [8]:
# --- BLOQUE 2: CARGA DE DATOS ---
# Justificación (Data Understanding): Carga eficiente eliminando nulos 
# que podrían romper los transformadores de Scikit-Learn.
print(" Cargando dataset procesado de la Fase 1...")
PATH_DATA = "data/office_ready_1M.csv.gz"

df = pd.read_csv(PATH_DATA, usecols=['text_cleaned', 'sentiment_class'])
initial_count = len(df)
df = df.dropna(subset=['text_cleaned']) 
df = df[df['text_cleaned'].str.strip() != ""]

print(f" Limpieza de vacíos: Se eliminaron {initial_count - len(df)} filas.")
print(f" Registros listos: {len(df):,}")

📦 Cargando dataset procesado de la Fase 1...
🧹 Limpieza de vacíos: Se eliminaron 1426 filas.
✅ Registros listos: 749,918


In [9]:
# --- BLOQUE 3: DIVISIÓN DE DATOS (PREVENCIÓN DE LEAKAGE) ---
# Justificación (Data Preparation): Dividimos ANTES de vectorizar. 
# Si vectorizamos antes de dividir, el vocabulario del set de prueba 
# "contaminaría" el entrenamiento (Data Leakage).
print(" Dividiendo datos en Train y Test (80/20)...")
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['text_cleaned'], df['sentiment_class'], 
    test_size=0.20, random_state=42, stratify=df['sentiment_class']
)

# Liberar RAM
del df
gc.collect()

✂️ Dividiendo datos en Train y Test (80/20)...


20

In [10]:
# --- BLOQUE 4: VECTORIZACIÓN TF-IDF ---
# Justificación (Data Preparation): Usamos min_df=5 para ignorar palabras
# que aparecen menos de 5 veces (errores ortográficos), reduciendo ruido.
print(" Entrenando TF-IDF Vectorizer...")
vectorizer = TfidfVectorizer(
    max_features=15000, 
    ngram_range=(1, 2), 
    stop_words='english',
    min_df=5 
)

# Fit SOLO en train, transform en test
X_train = vectorizer.fit_transform(X_train_text.astype(str))
X_test = vectorizer.transform(X_test_text.astype(str))

print(f" Matriz de entrenamiento: {X_train.shape}")

🧮 Entrenando TF-IDF Vectorizer...
✅ Matriz de entrenamiento: (599934, 15000)


In [11]:
# --- BLOQUE 5: ENTRENAMIENTO DE MODELOS MANUALES (CRISP-DM: Modeling) ---
# Justificación: Entrenamos un baseline simple (LogReg), un modelo geométrico
# de alto margen (LinearSVC) y un XGBoost (XGBoost).

print("\n Entrenando Modelo 1: Regresión Logística (Baseline)...")
log_model = LogisticRegression(max_iter=1000, n_jobs=-1, C=0.5) 
log_model.fit(X_train, y_train)
y_pred_log = log_model.predict(X_test)


🏃 Entrenando Modelo 1: Regresión Logística (Baseline)...


e:\ML-F1\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


In [12]:
print(" Entrenando Modelo 2: LinearSVC...")
svc_model = LinearSVC(dual=False, max_iter=2000, C=0.5)
svc_model.fit(X_train, y_train)
y_pred_svc = svc_model.predict(X_test)

⚔️ Entrenando Modelo 2: LinearSVC...


In [14]:
gc.collect()

print(" Entrenando Modelo 3: XGBoost (RTX 4070)...")

# 2. Configuración optimizada para XGBoost 2.0+ y GPUs NVIDIA modernas
xgb_model = xgb.XGBClassifier(
    tree_method='hist',     # Método de histograma, obligatorio para GPU
    device='cuda',          # Parámetro moderno unificado para usar la GPU
    n_estimators=600,       
    max_depth=5,            
    learning_rate=0.05,     
    objective='multi:softprob',
    num_class=3,
    random_state=42,
    # Sugerencia: si sigue fallando la memoria, añade: max_bin=64
)

# 3. Entrenamiento
# Nota: Si X_train es muy pesado, XGBoost lo moverá automáticamente a la VRAM
xgb_model.fit(X_train, y_train)

# 4. Predicción
y_pred_xgb = xgb_model.predict(X_test)

print(" Entrenamiento completado en GPU.")

🔥 Entrenando Modelo 3: XGBoost (RTX 4070)...


e:\ML-F1\.venv\Lib\site-packages\xgboost\core.py:751: UserWarning: [15:00:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


✅ Entrenamiento completado en GPU.


In [ ]:
# --- BLOQUE 6: BENCHMARK  CON AUTOML (PYCARET) ---
# Justificación: La se prioriza la viabilidad computacional.
# Para evitar colapsar los 16GB de RAM, tomamos una muestra estratificada de 
# 20,000 registros y las top 1000 features para el benchmark de AutoML.
print("\n Iniciando Benchmark AutoML (PyCaret)...")
from pycaret.classification import setup, compare_models

# Preparación de la muestra
sample_idx = np.random.choice(X_train.shape[0], 20000, replace=False)
X_dense_sample = X_train[sample_idx, :1000].toarray()
y_sample = y_train.iloc[sample_idx].values

df_automl = pd.DataFrame(X_dense_sample, columns=[f"feat_{i}" for i in range(1000)])
df_automl['sentiment_class'] = y_sample

# Setup silencioso de PyCaret
clf_setup = setup(
    data=df_automl, 
    target='sentiment_class', 
    session_id=42, 
    verbose=False,
    n_jobs=-1
)
# Comparamos modelos rápidos para el benchmark
best_automl = compare_models(include=['lr', 'dt', 'rf', 'lightgbm'], sort='Accuracy', n_select=1)

print("\n Mejor Modelo de AutoML:")
print(best_automl)

In [ ]:
# --- BLOQUE 7: EVALUACIÓN Y COMPARACIÓN FAIR (CRISP-DM: Evaluation) ---
# Justificación: Misma partición train/test. 
print("\n" + "="*40)
print(" COMPARACIÓN DE RESULTADOS FINALES")
print("="*40)

models = {
    "Logistic Regression (Baseline)": y_pred_log,
    "LinearSVC": y_pred_svc,
    "XGBoost (Manual)": y_pred_xgb
}

best_manual_acc = 0
best_manual_preds = None

for name, preds in models.items():
    acc = accuracy_score(y_test, preds)
    print(f"{name}: Accuracy = {acc:.4f}")
    if acc > best_manual_acc:
        best_manual_acc = acc
        best_manual_preds = preds